# Voice Assistant Cooking Bot
In this section, we will build **SugoGPT**, a voice-enabled chatbot assistant focused on answering recipe-related queries. The assistant will use a **Retrieval-Augmented Generation** (RAG) architecture to enhance its responses with relevant external information.

We will:

* Use the tokenizer previously trained during the fine-tuning phase.

* Compare the performance of the language model *with* and *without* RAG, to assess the impact of retrieval on response quality.

* Build a simple user interface that allows users to interact with the assistant through natural language queries.

# Setup
First, intall the required libraries.

In [1]:
!pip install -q openai-whisper gtts transformers langchain-community langchain-huggingface ffmpeg-python evaluate nltk rouge_score bert_score faiss-cpu --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 16.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Import general libraries.

In [2]:
# Disable Weights & Biases tracking
import os
os.environ["WANDB_DISABLED"] = "true"

# Core Libraries
import io
import time
import random
import ast
import base64
import json
import re
from typing import AnyStr

# Data Handling
import pandas as pd
import numpy as np

# Audio and Display
import torch

# Progress Bar
from tqdm import tqdm

## Load dataset

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/rcp-nlp/Mauro

/content/drive/.shortcut-targets-by-id/1ii6eI76RNWTvQduHWw9n3IAzswUbvxRG/rcp-nlp/Mauro


Function to load the data if FAISS index does not exist yet.

In [5]:
def load_data(filepath, sample_size=None):
    """
    Load the RecipeNLG dataset from CSV

    Args:
        filepath: Path to the full_dataset.csv
        sample_size: Number of recipes to sample (None to load all)

    Returns:
        pandas DataFrame with the recipe data
    """
    start_time = time.time()

    # Define the converters to parse stringified lists
    converters = {
        'ingredients': ast.literal_eval,
        'directions': ast.literal_eval,
        'ner': ast.literal_eval
    }

    # For a quick analysis, we can load a sample of the data
    if sample_size is not None:
        # Count the total number of lines in the file
        with open(filepath, 'r', encoding='utf-8') as f:
            line_count = sum(1 for _ in f)

        # Generate random line indices to sample
        skip_indices = sorted(random.sample(range(1, line_count), line_count - sample_size - 1))
        df = pd.read_csv(filepath, skiprows=skip_indices, converters=converters)
    else:
        # Load the full dataset
        df = pd.read_csv(filepath, converters=converters)

    # Ensure proper types
    # df['id'] = df['id'].astype(int)

    print(f"Data loaded in {time.time() - start_time:.2f} seconds")
    print(f"Dataset shape: {df.shape}")

    return df

## Create FAISS Vector Index from Documents

Each recipe in the DataFrame is wrapped in `Document` objects to prepare them for embedding and indexing. It formats relevant information such as title, ingredients, directions, cuisine, and course into a structured text block.

> *Note:* we only sample 100000 recipes.



In [6]:
from langchain.schema import Document

def create_documents(df):

  documents = []

  for _, row in tqdm(df.iterrows(), total=len(df)):
      recipe_text = f"""
      Title: {row['title']}
      Ingredients: {', '.join(row['ingredients'])}
      Directions: {' '.join(row['directions'])}
      Cuisine: {row.get('cuisine', 'Unknown')}
      Course: {row.get('course', 'Unknown')}
      """
      documents.append(Document(page_content=recipe_text.strip()))

  return documents

We start by initializing a pre-trained Hugging Face embedding model (`all-MiniLM-L6-v2`) to generate vector embeddings for our documents.

If a FAISS index already exists on disk, we load it to avoid recomputing embeddings and speed up the process. Otherwise, we load the dataset, create document objects, generate embeddings for a sample of the documents, build a new FAISS index, and save it locally for future use.

This FAISS index enables fast similarity search, which is a core component in the RAG pipeline.

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

index_path = "faiss_index_folder"
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(index_path):
    # Load existing FAISS index
    vectorstore = FAISS.load_local(index_path, embedding_model, allow_dangerous_deserialization=True)
    print("\nLoaded FAISS index from disk.")
else:

  # Adjust this path to where you have stored the dataset in Colab
  DATASET_PATH = '/content/drive/MyDrive/rcp-nlp/full_dataset.csv'

  df = load_data(DATASET_PATH, sample_size=10000)
  # df = load_data(DATASET_PATH, sample_size=None)

  # Create documents
  documents = create_documents(df)

  # Create new FAISS index from documents and save it
  vectorstore = FAISS.from_documents(documents, embedding_model)
  vectorstore.save_local(index_path)
  print("Created and saved new FAISS index.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Loaded FAISS index from disk.


The documents retrieved from the vectorstore are stored in a dictionary format, where each key is a unique document ID and each value is a Document object containing the content.

In [8]:
if 'documents' not in globals():
  documents = vectorstore.docstore._dict  # returns a dict with {doc_id: Document}

# To list them:
for doc_id, doc in list(documents.items())[:5]:
    print(f"ID: {doc_id}")
    print(f"Content: {doc.page_content[:300]}...\n")  # Print first 300 chars

ID: 1de3b0ba-7c10-4d45-8b81-d2a4bf413066
Content: Title: Potato And Cheese Pie
      Ingredients: 3 eggs, 1 tsp. salt, 1/4 tsp. pepper, 2 c. half and half, 3 c. potatoes, shredded coarse, 1 c. Cheddar cheese, coarsely shredded, 1/3 c. green onions
      Directions: Beat eggs, salt and pepper until well blended. Stir in half and half, potatoes and o...

ID: 52d6339a-e514-48c3-9395-6e63ad6370b4
Content: Title: Zucchini In Tomato Juice(From Weight Watchers)  
      Ingredients: zucchini, 12 oz. tomato juice, dash of oregano, dash of parsley flakes, dash of garlic powder, 2 Tbsp. bell pepper, 2 Tbsp. dehydrated onions, salt, 1 chicken bouillon cube
      Directions: Cut zucchini in half lengthwise, t...

ID: d6c974ab-4d4f-49d3-b22a-d52900d5b85e
Content: Title: Scalloped Corn
      Ingredients: dash of pepper, 1 beaten egg, 1/2 cup milk, 1 - 8-3/4 ounce can cream-style corn, 1 - 7 ounce can whole kernel corn, drained
      Directions: Combine ingredients and pour into 1 quart casserole dish

To support the construction of our Retrieval-Augmented Generation (RAG) pipeline, we first load the pre-trained `meta-llama/Llama-3.2-1B-Instruct` model from the Hugging Face Transformers library.

In [9]:
# Colab-specific
from google.colab import userdata

# Hugging Face Hub
from huggingface_hub import login

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline
)

from langchain.llms import HuggingFacePipeline


token = userdata.get('HF_TOKEN')
login(token=token)

model_name = "meta-llama/Llama-3.2-1B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create the generation pipeline
gen_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        use_cache=True,
        device_map="auto",
        max_length=2500,
        truncation=True,
        do_sample=True,
        top_k=5,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
)

generation_args = {
    "max_new_tokens": 200,
    "return_full_text": False,
    "temperature": 0.3,
    "do_sample": True,
}

# Wrap it for LangChain
llm = HuggingFacePipeline(pipeline=gen_pipeline, model_kwargs=generation_args)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Device set to use cuda:0
<ipython-input-9-932752de1514>:54: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=gen_pipeline, model_kwargs=generation_args)


Then, we set up the RAG pipeline by combining our language model with a retriever over the recipe knowledge base, using FAISS technology. This will help the model create answers based on the most similar recipes retrieved from the indexed data.

In [10]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA


# Define a friendly and expert cooking assistant prompt with RAG
cooker_prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
        You are SugoGPT, a friendly and expert virtual kitchen assistant trained on a wide variety of recipes, ingredients, and techniques from the RecipeNLG dataset.

        A user is asking you a question about cooking. You can reference the context retrieved from your recipe knowledge base. Give a helpful, conversational, and warm response as if you're speaking directly to them in their kitchen.
        Please provide a concise, single helpful answer, no additional follow-ups or dialogue.

        Context:
        {context}

        User's Question:
        {question}

        SugoGPT (your reply):
        """
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    chain_type="stuff",
    chain_type_kwargs={"prompt": cooker_prompt_template}
)

While, here, we define a baseline language model chain without retrieval, using a simple prompt that allows SugoGPT to generate answers solely based on its internal knowledge.

This no-RAG setup will serve as a comparison point to evaluate the benefits of adding retrieval later.

In [11]:
from langchain.chains import LLMChain

# Define the no-context prompt
cooker_prompt_template = PromptTemplate(
    input_variables=["question"],
    template="""
        You are SugoGPT, a friendly and expert virtual kitchen assistant trained on a wide variety of recipes, ingredients, and techniques from the RecipeNLG dataset.

        A user is asking you a question about cooking. Give a helpful, conversational, and warm response as if you're speaking directly to them in their kitchen.
        Please provide a concise, single helpful answer, no additional follow-ups or dialogue.

        User's Question:
        {question}

        SugoGPT (your reply):
        """
)

# Create the no-RAG chain using LLMChain
qa_no_rag = LLMChain(
    llm=llm,
    prompt=cooker_prompt_template
)

<ipython-input-11-4b845eecc7ef>:20: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  qa_no_rag = LLMChain(


In the following cell, we print the responses side-by-side to compare how the model answers the same cooking-related question both without and with the RAG pipeline.

In [12]:
question = "What are the ingredients for the 'Potato And Cheese Pie' recipe?"
response_no_rag = qa_no_rag.invoke(question)
response_rag = qa.run(question)

print("Without RAG:\n", response_no_rag["text"])
print("\n","="*100)
print("\n\nWith RAG:\n", response_rag)

<ipython-input-12-8506dfe4961f>:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response_rag = qa.run(question)


Without RAG:
 
        You are SugoGPT, a friendly and expert virtual kitchen assistant trained on a wide variety of recipes, ingredients, and techniques from the RecipeNLG dataset.

        A user is asking you a question about cooking. Give a helpful, conversational, and warm response as if you're speaking directly to them in their kitchen.
        Please provide a concise, single helpful answer, no additional follow-ups or dialogue.

        User's Question:
        What are the ingredients for the 'Potato And Cheese Pie' recipe?

        SugoGPT (your reply):
         "You're looking for the ingredients for the Potato And Cheese Pie recipe! I've got you covered. You'll need 2 large potatoes, 1 cup grated cheddar cheese, 1/2 cup all-purpose flour, 1/2 cup cold unsalted butter, 1/2 cup heavy cream, salt, and pepper. Let me know if you need any substitutions or variations, and I'll be happy to help!"



With RAG:
 
        You are SugoGPT, a friendly and expert virtual kitchen assista

Since we need to evaluate our model’s ability to generate accurate recipe answers, we prepare a small test dataset by extracting titles and recipes from the stored documents. For each recipe, we create a question asking for the recipe by its title and keep the corresponding ground truth recipe text.

In [13]:
import nltk
# nltk.download('punkt_tab')


def extract_title_and_recipe(text):
    lines = text.splitlines()
    title = None
    recipe_lines = []
    capture = False

    for line in lines:
        if line.startswith("Title:"):
            title = line[len("Title:"):].strip()
            capture = True  # Start capturing recipe lines after title
        elif capture:
            recipe_lines.append(line)

    recipe = "\n".join(recipe_lines).strip()
    return title, recipe

questions = []
ground_truth_recipes = []

# Only using 30 recipes:
for doc_id, doc in list(documents.items())[:30]:
    title, recipe = extract_title_and_recipe(doc.page_content)
    questions.append("What's the recipe of " + title + "?")
    ground_truth_recipes.append(recipe)

In [14]:
# Print first three recipes
questions[:3], ground_truth_recipes[:3]

(["What's the recipe of Potato And Cheese Pie?",
  "What's the recipe of Zucchini In Tomato Juice(From Weight Watchers)?",
  "What's the recipe of Scalloped Corn?"],
 ['Ingredients: 3 eggs, 1 tsp. salt, 1/4 tsp. pepper, 2 c. half and half, 3 c. potatoes, shredded coarse, 1 c. Cheddar cheese, coarsely shredded, 1/3 c. green onions\n      Directions: Beat eggs, salt and pepper until well blended. Stir in half and half, potatoes and onions. Pour into well-greased 8-inch baking dish. Bake in a 400° oven for 35 to 40 minutes, or until knife inserted in center comes out clean and potatoes are tender. Cool on rack 5 minutes; cut into squares. Makes 4 large servings.\n      Cuisine: Unknown\n      Course: Unknown',
  'Ingredients: zucchini, 12 oz. tomato juice, dash of oregano, dash of parsley flakes, dash of garlic powder, 2 Tbsp. bell pepper, 2 Tbsp. dehydrated onions, salt, 1 chicken bouillon cube\n      Directions: Cut zucchini in half lengthwise, then cut into cubes. Put all ingredients i

Next, we run a small evaluation loop over the test questions to collect responses from both the baseline model without retrieval and the RAG-enhanced model.

In [15]:
responses_no_rag = []
responses_rag = []


for question in tqdm(questions, total=len(questions), desc="Processing questions"):

  response_no_rag = qa_no_rag.invoke(question)
  actual_response_no_rag = response_no_rag["text"].split("SugoGPT (your reply):")[-1].strip()
  responses_no_rag.append(actual_response_no_rag)

  response_rag = qa.run(question)
  actual_response_rag = response_rag.split("SugoGPT (your reply):")[-1].strip()
  responses_rag.append(actual_response_rag)

Processing questions: 100%|██████████| 30/30 [08:51<00:00, 17.72s/it]


Now, we compute and compare automatic evaluation metrics for both versions, using the corresponding ground truth recipes as reference. Specifically, we calculate:

* **BLEU**: measures how closely the generated recipes match the reference text in terms of n-gram overlap

* **ROUGE**: captures recall-based overlap between the generated and reference texts (including ROUGE-1, ROUGE-2, and ROUGE-L)

* **METEOR**: evaluates the generated text by considering exact word matches, stemming, synonymy, and word order, offering a more flexible and semantically aware measure of similarity compared to BLEU

* **BERT SCORE**: assesses the semantic similarity between the generated and reference texts using contextualized embeddings from a pretrained BERT model. Rather than relying solely on surface-level matches, BERTScore captures deeper meaning by comparing token-level embeddings, making it particularly effective for evaluating fluency and semantic preservation in open-ended generation tasks.

In [16]:
import evaluate


# Load metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [17]:
def average(scores):
    return sum(scores) / len(scores) if scores else 0


bleu_rag_scores = []
bleu_no_rag_scores = []

rouge_rag_scores = []
rouge_no_rag_scores = []

meteor_rag_scores = []
meteor_no_rag_scores = []

bertscore_rag_scores = []
bertscore_no_rag_scores = []

for pred_rag, pred_no_rag, reference in zip(responses_rag, responses_no_rag, ground_truth_recipes):

    # BLEU
    bleu_rag_scores.append(
        bleu.compute(predictions=[pred_rag.lower()], references=[[reference.lower()]])["bleu"]
    )
    bleu_no_rag_scores.append(
        bleu.compute(predictions=[pred_no_rag.lower()], references=[[reference.lower()]])["bleu"]
    )

    # ROUGE
    rouge_rag_scores.append(
        rouge.compute(predictions=[pred_rag.lower()], references=[[reference.lower()]])
    )
    rouge_no_rag_scores.append(
        rouge.compute(predictions=[pred_no_rag.lower()], references=[[reference.lower()]])
    )

    # METEOR
    meteor_rag_scores.append(
        meteor.compute(predictions=[pred_rag.lower()], references=[[reference.lower()]])["meteor"]
    )
    meteor_no_rag_scores.append(
        meteor.compute(predictions=[pred_no_rag.lower()], references=[[reference.lower()]])["meteor"]
    )

# BERTSCORE
bertscore_rag_results = bertscore.compute(predictions=[p.lower() for p in responses_rag],
                                         references=[[r.lower()] for r in ground_truth_recipes],
                                         lang="en")
bertscore_no_rag_results = bertscore.compute(predictions=[p.lower() for p in responses_no_rag],
                                            references=[[r.lower()] for r in ground_truth_recipes],
                                            lang="en")

bertscore_rag_scores = bertscore_rag_results["f1"]
bertscore_no_rag_scores = bertscore_no_rag_results["f1"]

# Prepare average scores dictionary
metrics = {
    "BLEU": [average(bleu_no_rag_scores), average(bleu_rag_scores)]
}

# ROUGE returns dicts, average each key separately
rouge_keys = rouge_no_rag_scores[0].keys()
for key in rouge_keys:
    no_rag_avg = average([score[key] for score in rouge_no_rag_scores])
    rag_avg = average([score[key] for score in rouge_rag_scores])
    metrics[key.upper()] = [no_rag_avg, rag_avg]

metrics["METEOR"] = [average(meteor_no_rag_scores), average(meteor_rag_scores)]
metrics["BERTScore"] = [average(bertscore_no_rag_scores), average(bertscore_rag_scores)]

# Create DataFrame
df = pd.DataFrame(metrics, index=["No-RAG", "RAG"])

print("\n")
# Display rounded results
df.round(4)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,BLEU,ROUGE1,ROUGE2,ROUGEL,ROUGELSUM,METEOR,BERTScore
No-RAG,0.0245,0.2143,0.0421,0.1263,0.1502,0.1828,0.8230
RAG,0.1808,0.3966,0.2193,0.3220,0.3365,0.3557,0.8527


# Launch Voice Assistant Interface

Once the model is set up, we can run the script below to power an interactive voice assistant that combines speech recognition, language processing, and speech synthesis, enabling a fully conversational experience through voice input and output.

Here's what it covers:

- **UI Setup**: Loads a custom audio widget from an external HTML file to handle voice recording and playback with visual effects.
- **Voice Recording**: Uses browser-based JavaScript to record audio from the microphone, then decodes and saves it.
- **Audio Processing**: Converts recorded `.webm` audio to `.wav` format using FFmpeg for compatibility with OpenAI's Whisper model.
- **Speech Recognition**: Uses Whisper to transcribe the user's voice input to text.
- **Logging**: Saves both user and assistant messages into a `conversation_log.json` file with timestamps.
- **Language Response**: Passes the transcribed query to a QA system (`qa.run()`) for generating answers.
- **Speech Synthesis**: Converts the assistant's response to speech using Google Text-to-Speech (gTTS), encodes it to base64, and plays it in the browser using a visualized audio player.
- **Loop & Termination**: Repeats the interaction loop until the user says “exit”, “stop”, or “quit”.

This creates a fully voice-driven interface that allows users to interact with an AI assistant entirely by speaking.

In [18]:
from IPython.display import display, HTML, Audio, clear_output
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import whisper
from gtts import gTTS
import ffmpeg
import json
import datetime
import time


# Read the HTML/JS content from the file to use in the vocal assistant UI
with open("AUDIO.html", "r", encoding="utf-8") as f:
    AUDIO_HTML = f.read()

def save_conversation(speaker, text):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = {
        "timestamp": timestamp,
        "speaker": speaker,
        "text": text
    }

    # Load existing conversation
    conversation = []
    if os.path.exists('conversation_log.json'):
        with open('conversation_log.json', 'r') as f:
            try:
                conversation = json.load(f)
            except json.JSONDecodeError:
                conversation = []

    # Append new entry
    conversation.append(entry)

    # Save updated conversation
    with open('conversation_log.json', 'w') as f:
        json.dump(conversation, f, indent=2)

def record_audio():
    display(HTML(AUDIO_HTML))
    audio_data = eval_js("window._colabRecordingPromise")
    audio_bytes = b64decode(audio_data.split(',')[1])

    with open('input_audio.webm', 'wb') as f:
        f.write(audio_bytes)

    # Convert webm to wav for Whisper
    try:
        (
            ffmpeg
            .input('input_audio.webm')
            .output('input_audio.wav', format='wav', acodec='pcm_s16le', ac=1, ar='16000')
            .overwrite_output()
            .run(quiet=True)
        )
        os.remove('input_audio.webm')  # remove webm after conversion
        return 'input_audio.wav'
    except Exception as e:
        print("Audio conversion error:", e)
        return 'input_audio.webm'  # fallback

def load_whisper_model(model_size="base"):
    # Load silently without printing
    return whisper.load_model(model_size)

def transcribe_audio(model, audio_file):
    result = model.transcribe(audio_file)
    transcription = result["text"]
    save_conversation("User", transcription)
    return transcription

def wait_for_audio_complete():
    """Wait for audio playback to complete by checking the flag element."""

    while True:
        flag_content = eval_js("document.getElementById('audio-complete-flag').textContent")
        if flag_content == 'complete' or flag_content == 'error':
            if flag_content == 'error':
                print("Audio playback encountered an error")
            break
    # Reset the flag
    eval_js("document.getElementById('audio-complete-flag').textContent = ''")

def speak_text(text):
    if not text.strip():
        return

    # Save bot's response to conversation log
    save_conversation("SugoGPT", text)

    # Generate audio file with slightly slower speed for better clarity
    tts = gTTS(text=text, lang='en', slow=False)
    tts.save("response.mp3")

    # Convert to base64 for JavaScript playback
    with open("response.mp3", "rb") as audio_file:
        audio_base64 = b64encode(audio_file.read()).decode('utf-8')

    # Use JavaScript to play the audio with visualization
    audio_data_uri = f"data:audio/mp3;base64,{audio_base64}"
    eval_js(f'window.playAudioResponse("{audio_data_uri}")')

    # Wait for audio playback to complete
    wait_for_audio_complete()

# Display the UI only once at the beginning
def initialize_ui():

    # Load model once
    print("Loading Whisper model...")
    whisper_model = load_whisper_model("base")
    print("Voice assistant ready! Click the recording button to start.")

    # Clear any previous output
    clear_output(wait=True)
    # Clear the conversation log at the start of each run
    with open('conversation_log.json', 'w') as f:
        f.write('[]')
    # Display the UI HTML
    display(HTML(AUDIO_HTML))
    bot_presentation = "Hi there! I’m SugoGPT, your friendly virtual chef assistant. What delicious recipe can I help you with today?"
    speak_text(bot_presentation)

# Main function
def start_voice_assistant(whisper_model, qa):
    # Initialize UI
    initialize_ui()

    exit_phrases = [
    "goodbye", "bye", "see you", "see you later",
    "talk to you later", "thanks that's all", "i'm done",
    "that's it", "no more questions", "catch you later"
    ]

    while True:
        # Make sure the UI is intact
        audio_file = record_audio()
        user_query = transcribe_audio(whisper_model, audio_file)

        # Process the user's query here
        response = qa.run(user_query)
        actual_response = response.split("SugoGPT (your reply):")[-1].strip()

        # Speak the response and wait for it to complete
        speak_text(actual_response)

        # Check for exit command
        if any(user_query == phrase or phrase in user_query for phrase in exit_phrases):
          print("Voice assistant terminated.")
          break

In [19]:
whisper_model = load_whisper_model("base")

import warnings
warnings.filterwarnings('ignore')

100%|███████████████████████████████████████| 139M/139M [00:04<00:00, 30.3MiB/s]


The assistant is now ready to accept and process voice input.

In [23]:
start_voice_assistant(whisper_model, qa)

Voice assistant terminated.


Let’s visualize the conversation history using the stored JSON and embed it into the chat UI.

In [34]:
with open('conversation_log.json', 'r') as f:
    conversation_data = f.read()

# Parse the JSON data
messages = json.loads(conversation_data)

# Get HTML for chat layout
with open('chat_UI.html', 'r') as f:
    chat_UI = f.read()

# Add chat history
for msg in messages:
    speaker_class = "user" if msg["speaker"].lower() == "user" else "bot"
    chat_UI += f'''
    <div class="message {speaker_class}">
      <strong>{msg["speaker"]}</strong>
      <p>{msg["text"]}</p>
      <span class="timestamp">{msg["timestamp"]}</span>
    </div>
    '''

chat_UI += '</div>'

# Display the HTML
display(HTML(chat_UI))

In [30]:
messages = json.loads(conversation_data)
messages

[{'timestamp': '2025-05-24 17:52:36',
  'speaker': 'SugoGPT',
  'text': 'Hi there! I’m SugoGPT, your friendly virtual chef assistant. What delicious recipe can I help you with today?'},
 {'timestamp': '2025-05-24 17:52:56',
  'speaker': 'User',
  'text': ' Hi there, do you know how to make some pizza?'},
 {'timestamp': '2025-05-24 17:53:03',
  'speaker': 'SugoGPT',
  'text': "Ah, you're looking to make a pizza! I'd be happy to help you with that. To get started, you'll need to make the pizza dough. You can either use a recipe like the one I provided earlier or try a simple one using a bread machine. For a basic pizza dough, you'll need 1 package of dry active yeast, 1 teaspoon of sugar, 1 cup of warm water (around 120F), 18 cups of extra virgin olive oil, 1 teaspoon of salt, and 3 cups of all-purpose flour. Mix everything together, let it rise for 45 minutes, and then divide it into 3-4 portions. You can shape the dough into balls and let them rise again for a few more minutes before b